In [ ]:
from pyspark.sql import functions as F
import pandas as pd

In [ ]:
orders_path = "orders.json"
products_path = "products.csv"
customers_path = "customer.xlsx"

In [ ]:

orders_raw = (
    spark.read
    .option("multiline", "true")
    .json(orders_path)
)
products_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(products_path)
)


customers_pd = pd.read_excel(customers_path)
customers_raw = spark.createDataFrame(customers_pd)

display(orders_raw.limit(5))
display(products_raw.limit(5))
display(customers_raw.limit(5))

In [ ]:
orders_clean = (
    orders_raw
    .withColumn("Order_Date", F.to_date(F.col("Order Date"), "d/M/yyyy"))
    .withColumn("Ship_Date", F.to_date(F.col("Ship Date"), "d/M/yyyy"))
    .withColumn("Quantity", F.col("Quantity").cast("int"))
    .withColumn("Price", F.col("Price").cast("double"))
    .withColumn("Discount", F.col("Discount").cast("double"))
    .withColumn("Profit", F.round(F.col("Profit").cast("double"), 2))
    .dropDuplicates(["Row ID"])
    .select(
        F.col("Row ID").alias("Row_ID"),
        F.col("Order ID").alias("Order_ID"),
        "Order_Date",
        "Ship_Date",
        F.col("Ship Mode").alias("Ship_Mode"),
        F.col("Customer ID").alias("Customer_ID"),
        F.col("Product ID").alias("Product_ID"),
        "Quantity",
        "Price",
        "Discount",
        "Profit"
    )
)
display(orders_clean)

In [ ]:
products_clean = (
    products_raw
    .withColumnRenamed("Product ID", "Product_ID")
    .withColumnRenamed("Sub-Category", "Sub_Category")
    .withColumnRenamed("Product Name", "Product_Name")
    .withColumnRenamed("Price per product", "Product_Price")
    .withColumn("Product_Price", F.col("Product_Price").cast("double"))
    .dropDuplicates(["Product_ID"])
    .select(
        "Product_ID",
        "Category",
        "Sub_Category",
        "Product_Name",
        "State",
        "Product_Price"
    )
)
display(products_clean)

In [ ]:
customers_clean = (
    customers_raw
    .withColumnRenamed("Customer ID", "Customer_ID")
    .withColumnRenamed("Customer Name", "Customer_Name")
    .withColumnRenamed("Postal Code", "Postal_Code")
    .dropDuplicates(["Customer_ID"])
    .select(
        "Customer_ID",
        "Customer_Name",
        "Segment",
        "Country",
        "City",
        "State",
        "Postal_Code",
        "Region"
    )
)
display(customers_clean)

In [ ]:
master_df = (
    orders_clean.alias("o")
    .join(
        customers_clean.alias("c"),
        F.col("o.Customer_ID") == F.col("c.Customer_ID"),
        "left"
    )
    .join(
        products_clean.alias("p"),
        F.col("o.Product_ID") == F.col("p.Product_ID"),
        "left"
    )
    .withColumn("Year", F.year(F.col("o.Order_Date")))
    .select(
        F.col("o.Row_ID"),
        F.col("o.Order_ID"),
        F.col("o.Order_Date"),
        F.col("o.Ship_Date"),
        F.col("o.Ship_Mode"),
        F.col("o.Customer_ID"),
        F.col("c.Customer_Name"),
        F.col("c.Segment"),
        F.col("c.Country").alias("Customer_Country"),
        F.col("c.City").alias("Customer_City"),
        F.col("c.State").alias("Customer_State"),
        F.col("c.Postal_Code"),
        F.col("c.Region"),
        F.col("o.Product_ID"),
        F.col("p.Product_Name"),
        F.col("p.Category"),
        F.col("p.Sub_Category"),
        F.col("p.State").alias("Product_State"),
        F.col("p.Product_Price"),
        F.col("o.Quantity"),
        F.col("o.Price"),
        F.col("o.Discount"),
        F.col("o.Profit"),
        F.col("Year")
    )
)
display(master_df.limit(10))

In [ ]:
profit_by_year = (
    master_df
    .groupBy("Year")
    .agg(F.round(F.sum("Profit"), 2).alias("Total_Profit"))
    .orderBy("Year")
)
display(profit_by_year)

In [ ]:
profit_by_year_category = (
    master_df
    .groupBy("Year", "Category")
    .agg(F.round(F.sum("Profit"), 2).alias("Total_Profit"))
    .orderBy("Year", "Category")
)
display(profit_by_year_category)

In [ ]:

profit_by_customer = (
    master_df
    .groupBy("Customer_ID", "Customer_Name")
    .agg(F.round(F.sum("Profit"), 2).alias("Total_Profit"))
    .orderBy(F.desc("Total_Profit"))
)
display(profit_by_customer)

In [ ]:

profit_by_customer_year = (
    master_df
    .groupBy("Customer_ID", "Customer_Name", "Year")
    .agg(F.round(F.sum("Profit"), 2).alias("Total_Profit"))
    .orderBy("Customer_ID", "Year")
)
display(profit_by_customer_year)

In [ ]:
#writing this for test cases
orders_clean.write.mode("overwrite").format("delta").saveAsTable("raw_orders")
products_clean.write.mode("overwrite").format("delta").saveAsTable("raw_products")
customers_clean.write.mode("overwrite").format("delta").saveAsTable("raw_customers")
master_df.write.mode("overwrite").format("delta").saveAsTable("master_sales")
profit_by_year.write.mode("overwrite").format("delta").saveAsTable("profit_by_year")
profit_by_year_category.write.mode("overwrite").format("delta").saveAsTable("profit_by_year_category")
profit_by_customer.write.mode("overwrite").format("delta").saveAsTable("profit_by_customer")
profit_by_customer_year.write.mode("overwrite").format("delta").saveAsTable("profit_by_customer_year")

#view to execute sql queries
master_df.createOrReplaceTempView("master_sales")

In [ ]:
#SQL QUERIES
Profit by Year

In [ ]:
%sql
SELECT
    Year,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM master_sales
GROUP BY Year
ORDER BY Year;


Profit by Year + Product Category

In [ ]:
%sql
SELECT
    Year,
    Category,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM master_sales
GROUP BY Year, Category
ORDER BY Year, Category;

Profit by Customer

In [ ]:
%sql
SELECT
    Customer_ID,
    Customer_Name,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM master_sales
GROUP BY Customer_ID, Customer_Name
ORDER BY Total_Profit DESC;

Profit by Customer + Year

In [ ]:
%sql
SELECT
    Customer_ID,
    Customer_Name,
    Year,
    ROUND(SUM(Profit), 2) AS Total_Profit
FROM master_sales
GROUP BY Customer_ID, Customer_Name, Year
ORDER BY Customer_ID, Year;